# 00 · Playground: try the src/ code yourself
Kernel: **bn4601a** (laptop). Run the cells top to bottom with **Shift+Enter**. Change things and re-run, nothing here can break the project.

## Step 1: tell Python where our code is (run this first, every time)

In [1]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"          # this notebook is in reasoning/notebooks, the code is in reasoning/src
sys.path.insert(0, str(SRC))
print("our code is in:", SRC)

our code is in: C:\Users\likin\Documents\BN4601A\group project\BN4601A\reasoning\src


## Step 2: config.py (Lesson 1)

In [2]:
import config
print("project folder:", config.REASONING)
print("data folder:   ", config.PROCESSED)
print("model:         ", config.BASE_MODEL, "| seed:", config.SEED)

project folder: C:\Users\likin\Documents\BN4601A\group project\BN4601A\reasoning
data folder:    C:\Users\likin\Documents\BN4601A\group project\BN4601A\reasoning\data\processed
model:          google/medgemma-1.5-4b-it | seed: 42


## Step 3: data.py (Lesson 2)

In [3]:
from data import load_pool1

test_ktas = load_pool1(split="test", source="ktas")
print(len(test_ktas), "patients,", test_ktas["urgent"].sum(), "urgent")    # 190 patients, 37 urgent
test_ktas.head()

190 patients, 37 urgent


,source,id,split,age,sex,ambulance,injured,mental,chief_complaint,nrs_pain,sbp,dbp,hr,rr,temp_c,spo2,triage_level,urgent
0,ktas,ktas_1,test,71.0,male,0.0,1.0,alert,right ocular pain,2.0,160.0,100.0,84.0,18.0,36.6,100.0,4,0
1,ktas,ktas_2,test,56.0,female,0.0,1.0,alert,right forearm burn,2.0,137.0,75.0,60.0,20.0,36.5,NaN,5,0
2,ktas,ktas_8,test,78.0,male,0.0,0.0,alert,"pain, chest",3.0,169.0,86.0,80.0,20.0,36.0,NaN,3,0
3,ktas,ktas_14,test,55.0,male,0.0,0.0,alert,abd pain,3.0,121.0,78.0,80.0,20.0,36.4,NaN,4,0
4,ktas,ktas_15,test,58.0,male,0.0,0.0,alert,headache,3.0,135.0,82.0,106.0,20.0,37.6,NaN,4,0


In [4]:
# Try it: change "test" to "train" or "val", or "ktas" to "nhamcs", and re-run
everyone = load_pool1()
print("everyone:", len(everyone))

everyone: 9674


## Step 4: cards.py, the _num helper (Lesson 3a)

In [5]:
from cards import _num, make_card

print(_num(84.0))
print(_num(float("nan")))
print(_num(98.0, "{:g}%"))
print(_num(36.65, "{:.1f} C"))

84
not recorded
98%
36.6 C


## Step 5: make one card (without Caleb's vitals note: that's Lesson 3b)

In [6]:
patient = test_ktas.iloc[0]          # .iloc[0] = the first row (Python counts from 0)
print(make_card(patient))

=== ED PATIENT RECORD ===
PATIENT: 71 male | Arrival: walk-in / private transport | Time: not recorded
PRESENTING COMPLAINT: right ocular pain (injury)
HISTORY OF PRESENTING COMPLAINT: not recorded
PAST HISTORY: not recorded
MEDICATIONS: not recorded
ALLERGIES: not recorded
OBSERVATIONS: HR 84 | BP 160/100 | RR 18 | SpO2 100% (room air assumed) | Temp 36.6 C | Consciousness alert | Pain 2/10
VITALS ASSESSMENT: not available
IMAGING: none
CONSULTATION NOTES: none
=== END RECORD ===


In [7]:
# Try it: pick another patient (e.g. iloc[5]) and see how missing values show up
print(make_card(test_ktas.iloc[5]))

=== ED PATIENT RECORD ===
PATIENT: 32 male | Arrival: walk-in / private transport | Time: not recorded
PRESENTING COMPLAINT: post seizure
HISTORY OF PRESENTING COMPLAINT: not recorded
PAST HISTORY: not recorded
MEDICATIONS: not recorded
ALLERGIES: not recorded
OBSERVATIONS: HR 101 | BP 124/81 | RR 18 | SpO2 not recorded (room air assumed) | Temp 36.7 C | Consciousness alert | Pain 0/10
VITALS ASSESSMENT: not available
IMAGING: none
CONSULTATION NOTES: none
=== END RECORD ===


## Step 6: the finished cards for all 9,674 patients (already built, with Caleb's vitals note)

In [8]:
import pandas as pd
cards = pd.read_parquet(config.PROCESSED / "cards_pool1.parquet")
print(len(cards), "cards")
print(cards.iloc[0]["card"])

9674 cards
=== ED PATIENT RECORD ===
PATIENT: 71 male | Arrival: walk-in / private transport | Time: not recorded
PRESENTING COMPLAINT: right ocular pain (injury)
HISTORY OF PRESENTING COMPLAINT: not recorded
PAST HISTORY: not recorded
MEDICATIONS: not recorded
ALLERGIES: not recorded
OBSERVATIONS: HR 84 | BP 160/100 | RR 18 | SpO2 100% (room air assumed) | Temp 36.6 C | Consciousness alert | Pain 2/10
VITALS ASSESSMENT: NEWS2 0 = LOW risk. All recorded vitals in normal range. Vitals model: 35% chance of needing KTAS 1-2 care (above the 14% alert threshold). Safety floor: priority 3 or more urgent.
IMAGING: none
CONSULTATION NOTES: none
=== END RECORD ===
